<a href="https://colab.research.google.com/github/hminh1805/LVLM_Interpretation/blob/main/LVLM_task4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Tạo thư mục trong Drive

In [25]:
import os
from google.colab import drive

# 1. Kết nối Drive
drive.mount('/content/drive')

# 2. Setup cấu trúc chuẩn của dự án
PROJECT_ROOT = '/content/drive/MyDrive/EyeCode_LVLM_Project'
CODE_DIR = os.path.join(PROJECT_ROOT, 'LVLM_Interpretation')
DATASET_DIR = os.path.join(PROJECT_ROOT, 'dataset')
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results')
PIP_CACHE = os.path.join(PROJECT_ROOT, 'pip_cache')

# 3. Tạo thư mục tự động
os.makedirs(os.path.join(DATASET_DIR, 'images'), exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(PIP_CACHE, exist_ok=True)

# 4. Ép cache pip vào Drive để tải lại nhanh ở lần sau
os.environ['PIP_CACHE_DIR'] = PIP_CACHE

print(f"Mọi thứ đã sẵn sàng tại: {PROJECT_ROOT}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Mọi thứ đã sẵn sàng tại: /content/drive/MyDrive/EyeCode_LVLM_Project


Lưu code về Drive và tải thư viện

In [26]:
%cd /content
if not os.path.exists(CODE_DIR):
    !git clone https://github.com/bytedance/LVLM_Interpretation.git
    !cp -r LVLM_Interpretation {CODE_DIR}
    print("Đã sao chép code sang Drive!")

%cd {CODE_DIR}

# Cài đặt thư viện (nhờ có pip_cache ở Cell 1 nên sẽ rất nhanh)
!pip install --upgrade pip
!pip install cmake sentencepiece deepspeed yake ezcolorlog
!pip install open_clip_torch einops ninja
!pip install transformers==4.37.2 peft==0.8.2 diffusers==0.26.3 huggingface-hub==0.25.2
!pip install triton==3.6.0 bitsandbytes==0.45.5 accelerate==0.27.2

# Vá lỗi import của tác giả
!find . -type f -name "*.py" -exec sed -i 's/transformers.deepspeed/transformers.integrations.deepspeed/g' {} +
print("--- CÀI ĐẶT THƯ VIỆN HOÀN TẤT ---")

/content
/content/drive/.shortcut-targets-by-id/1F_5wqRUVnvQ5MLoxU5cCu7Y14CUafu3L/EyeCode_LVLM_Project/LVLM_Interpretation
--- CÀI ĐẶT THƯ VIỆN HOÀN TẤT ---


Ép 4 bit


In [27]:
%cd {CODE_DIR}
import re

with open('main.py', 'r') as f:
    content = f.read()

# Chèn cờ ép cân 4-bit để GPU không bị nổ RAM
content = re.sub(
    r"load_pretrained_model\(model_path, args\.model_base, model_name\)",
    r"load_pretrained_model(model_path, args.model_base, model_name, load_in_4bit=True, device_map='auto')",
    content
)

# Vá lỗi thiết bị
content = content.replace(".half().cuda()", ".half().to(model.device)")

with open('main.py', 'w') as f:
    f.write(content)

print("Đã ép 4-bit thành công vào file main.py trên Drive!")

/content/drive/.shortcut-targets-by-id/1F_5wqRUVnvQ5MLoxU5cCu7Y14CUafu3L/EyeCode_LVLM_Project/LVLM_Interpretation
Đã ép 4-bit thành công vào file main.py trên Drive!


Tạo dataset (nhớ thêm ảnh vào thư mục trước khi chạy)

In [28]:
import json

dummy_data = [
    {
        "question_id": 1,
        "image": "dogs.png",
        "text": "How many animals are there in the image?"

    },
    {
        "question_id": 2,
        "image": "cow.png",
        "text": "What animal is this ?"
    },
    {
        "question_id": 3,
        "image": "lambo.jpg",
        "text": "What color and what brand is this car?"
    }
]

json_path = os.path.join(DATASET_DIR, 'questions.json')
with open(json_path, 'w') as f:
    json.dump(dummy_data, f, indent=4)

IMAGE_PATH = os.path.join(DATASET_DIR, 'images')
!ls -lh {IMAGE_PATH}
print(f"Đã tạo file câu hỏi chuẩn tại: {json_path}")


total 574K
-rw------- 1 root root  25K Apr 26 09:49 cat.jpg
-rw------- 1 root root 455K Apr 26 13:42 cow.png
-rw------- 1 root root  67K Apr 26 10:27 dogs.png
-rw------- 1 root root  28K Apr 26 08:44 lambo.jpg
Đã tạo file câu hỏi chuẩn tại: /content/drive/MyDrive/EyeCode_LVLM_Project/dataset/questions.json


RUN

In [29]:
  import os
  import torch

  # Dọn dẹp bộ nhớ trước khi chạy
  os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
  torch.cuda.empty_cache()

  %cd {CODE_DIR}

  # Chạy mô hình (tui chỉnh thông số iterations lên 5 và ig_iter 10 để ảnh Heatmap hiện ra sắc nét nhé)
  !BNB_CUDA_VERSION=121 LD_LIBRARY_PATH=$LD_LIBRARY_PATH:/usr/local/cuda/lib64 python3 main.py \
    --method iGOS+ \
    --model llava \
    --dataset llava-bench \
    --data_path "/content/drive/MyDrive/EyeCode_LVLM_Project/dataset/questions.json" \
    --image_folder "/content/drive/MyDrive/EyeCode_LVLM_Project/dataset/images" \
    --output_dir "/content/drive/MyDrive/EyeCode_LVLM_Project/results" \
    --size 32 --L1 1.0 --L2 0.1 --L3 10.0 --ig_iter 2 --gamma 1.0 --iterations 2 --momentum 5 --max_new_tokens 20

/content/drive/.shortcut-targets-by-id/1F_5wqRUVnvQ5MLoxU5cCu7Y14CUafu3L/EyeCode_LVLM_Project/LVLM_Interpretation
2026-04-26 14:47:37.584161: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777214857.606484   18305 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777214857.613710   18305 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777214857.632414   18305 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777214857.632445   18305 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the s